In [ ]:
### question: why does the paper average density matrix before entropy

In [66]:
import numpy as np
from qiskit.quantum_info import DensityMatrix, random_unitary
import matplotlib.pyplot as plt

def block_diagonal_unitary():
    """Generate a block-diagonal unitary matrix that leaves |↑↑⟩ invariant and acts randomly on |↑↓⟩, |↓↑⟩, |↓↓⟩."""
    U3 = random_unitary(3).data  # Haar-random unitary in the 3x3 subspace
    U = np.eye(4, dtype=complex)
    U[1:, 1:] = U3  # Applying the random unitary to the |↑↓⟩, |↓↑⟩, |↓↓⟩ states
    return U

def tensor_product_operators(ops):
    """Compute the tensor product of a list of operators."""
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def even_full_unitary(L):
    """Build the full unitary matrix for even steps by tensoring together the unitaries acting on even pairs of sites."""
    ops = [block_diagonal_unitary() for _ in range(L // 2)]
    return tensor_product_operators(ops)

def rotate_right(bit_string, k):
    """Rotate a bit string to the right by k positions."""
    n = len(bit_string)
    k = k % n
    return bit_string[-k:] + bit_string[:-k]

def odd_full_unitary(L):
    """Build the full unitary matrix for odd steps by rotating the indices of the even unitary."""
    U = even_full_unitary(L)
    new_U = np.zeros((2**L, 2**L), dtype=complex)

    for i in range(2**L):
        for j in range(2**L):
            new_i = int(rotate_right(f'{i:b}'.zfill(L), 1), 2)
            new_j = int(rotate_right(f'{j:b}'.zfill(L), 1), 2)
            new_U[new_i, new_j] = U[i, j]
    
    return new_U

def apply_custom_unitaries(state, L, step):
    """Apply custom block-diagonal unitaries on the system in a zigzag pattern."""
    if step % 2 == 0: # if the step is even
        # Apply unitaries to pairs (0-1, 2-3, 4-5, ...)
        U = even_full_unitary(L)
        return U @ state @ U.conj().T
    else:
        # Apply unitaries to pairs (1-2, 3-4, 5-0, ...)
        U = odd_full_unitary(L)
        return U @ state @ U.conj().T

def apply_measurement_feedback_operator(state, L, pm, pf):
    """Generate the measurement and feedback operator for the system."""
    I = np.eye(2)
    X = np.array([[0, 1], [1, 0]])
    P0 = np.array([[1, 0], [0, 0]])
    P1 = np.array([[0, 0], [0, 1]])

    state_density = state
    full_operator = np.eye(2**L, dtype=complex)
    names = ['I'] * L  # Initialize names for all qubits

    for i in range(L):
        # Create projection operators for the full state
        P0_full = np.kron(np.eye(2**i), np.kron(P0, np.eye(2**(L-i-1))))
        P1_full = np.kron(np.eye(2**i), np.kron(P1, np.eye(2**(L-i-1))))

        prob_0 = np.real(np.trace(P0_full @ state_density @ P0_full)) # the probability of measuring the i-th qubit to be up
        prob_1 = np.real(np.trace(P1_full @ state_density @ P1_full)) # the probability of measuring the i-th qubit to be down

        if np.random.rand() < pm: # if we are measuring the i-th qubit
            if np.random.rand() < prob_0: # if the i-th qubit is measured to be up
                state_density = P0_full @ state_density @ P0_full
                names[i] = 'P0'
            else:
                state_density = P1_full @ state_density @ P1_full
                names[i] = 'P1'
                if np.random.rand() < pf:
                    feedback_full = np.kron(np.eye(2**i), np.kron(X, np.eye(2**(L-i-1))))
                    state_density = feedback_full @ state_density @ feedback_full.conj().T
                    names[i] = 'XP1'
        state_density /= np.real(np.trace(state_density))

    full_name = ', '.join(names)
    return full_operator, full_name, np.real(state_density)

def second_renyi_entropy(rho):
    """
    Compute the second Rényi entropy for a given density matrix rho.

    Parameters:
    rho (np.ndarray): Density matrix

    Returns:
    float: Second Rényi entropy
    """
    return -np.log2(np.trace(rho @ rho))

def simulate_circuit(L, pm, pf, num_steps):
    """Simulate the circuit evolution for a given number of steps."""
    state = DensityMatrix.from_label('1' * L).data
    renyi_entropies = []
    trajectory = []

    for step in range(num_steps):
        # Apply custom unitaries
        state = apply_custom_unitaries(state, L, step)
        state_before = state
        # Apply measurement and feedback
        measurement_operator, name, state = apply_measurement_feedback_operator(state, L, pm, pf)
        trace = np.abs(np.trace(state))
        if np.isnan(trace):
            print(state_before)
        # Compute the second Rényi entropy
        renyi_entropy = second_renyi_entropy(state)
        renyi_entropies.append(renyi_entropy)

        trajectory.append(state)
        
    # Convert the final density matrix to probabilities
    probabilities = np.abs(np.diag(state))
    return probabilities, renyi_entropies, trajectory

def run_simulation(L_values, pm, pf, num_steps, num_runs, display_plots=True):
    """Run the simulation for multiple system sizes and plot the entanglement entropy."""
    all_entropies = {}
    all_trajectories = {}

    for L in L_values:
        print(f"Running simulation for L={L}")
        accumulated_counts = np.zeros(2**L)
        entropies = []
        trajectories = []

        for _ in range(num_runs):
            final_counts, renyi_entropies, trajectory = simulate_circuit(L, pm, pf, num_steps)
            accumulated_counts += final_counts
            entropies.append(renyi_entropies)
            trajectories.append(trajectory)

        # Normalize the accumulated counts
        accumulated_counts /= num_runs

        # Store the entropies and trajectories
        avg_entropies = np.mean(entropies, axis=0)
        avg_trajectories = np.mean(trajectories, axis=0)
        all_entropies[L] = avg_entropies
        all_trajectories[L] = avg_trajectories

        if display_plots:
            # Generate basis state labels
            basis_states = [bin(i)[2:].zfill(L) for i in range(2**L)]

            # Plot the final measurement results
            plt.figure()
            plt.bar(basis_states, accumulated_counts)
            plt.xlabel('Basis states')
            plt.ylabel('Probability')
            plt.title(f'Probability Distribution of Final States for L={L}')
            plt.xticks(rotation=90)
            plt.show()

    if display_plots:
        # Plot the entanglement entropy for different system sizes
        plt.figure()
        for L in L_values:
            plt.plot(range(num_steps), all_entropies[L] / L, marker='o', label=f'L={L}')  # Divide by L to normalize
        plt.xlabel('Step')
        plt.ylabel('Second Rényi Entropy per qubit')
        plt.title('Evolution of Second Rényi Entropy per Qubit for Different System Sizes')
        plt.legend()
        plt.show()

    return all_entropies, all_trajectories

# Parameters
L_values = [4, 6]  # Different system sizes
pm = 0.5  # Measurement rate
pf = 0.5  # Feedback rate
num_steps = 20  # Number of steps in the simulation
num_runs = 10  # Number of times to repeat the entire simulation

# Run the simulation and optionally display plots
all_entropies, all_trajectories = run_simulation(L_values, pm, pf, num_steps, num_runs, display_plots=False)

Running simulation for L=4
Running simulation for L=6
